# 31.03 Идентифицируемость двуслойной задачи

> **Статус:** канонический синтетический анализ локальной структурной
> идентифицируемости. Практическая идентифицируемость по экспериментам 2–3
> остаётся заблокированной до калибровки наблюдения и оценки раздельных
> составляющих неопределённости.

Ноутбук использует только идеальную плоскую двуслойную модель и не читает
данные добровольцев. Числа ниже не являются оценками тканей, погрешности
прибора или качества конкретной записи.


## Происхождение и исправление исторической логики

Источник — фрагменты Фишера/CRLB/SVD из старых документов `04`, `05`, `12`,
сохранённые в `archive/legacy/31.90_Объединённый_анализ_чувствительности.ipynb`.
Исторический код строил размерный якобиан $(\rho_1,h,\rho_2)$, задавал
$\Sigma=\sigma^2I$ и подставлял вместо $\sigma$ среднеквадратичный остаток
подгонки. Но этот остаток включал систематику размеров и ошибку самой модели;
его независимость, одинаковая дисперсия и гауссовость не были установлены.
Кроме того, сохранённых outputs расчёта нет.

Поэтому архивные значения стандартных отклонений и выводы о достаточности либо
недостаточности диапазона размеров **не считаются подтверждёнными**. Остаток
подгонки нельзя автоматически называть шумом и подставлять в CRLB. Ниже
восстановлена только часть, которая определяется самой прямой моделью:

1. ранг и сингулярные числа логарифмического якобиана;
2. точная неидентифицируемость пары при неизвестном $h$;
3. точная масштабная неидентифицируемость абсолютных $\rho$ при неизвестном
   multiplicative gain;
4. интерфейс условного CRLB, который не выдаёт результат без явно переданной
   ковариации относительных ошибок.


## Три разных уровня утверждений

### 1. Структурная локальная идентифицируемость

При известном из внешнего источника $h$, калиброванном тракте и двух разных
размерах используется безразмерный якобиан

$$
\widetilde J_{ij}=\begin{bmatrix}
S_{\rho_1}(L_i)&S_{\rho_2}(L_i)\\
S_{\rho_1}(L_j)&S_{\rho_2}(L_j)
\end{bmatrix}.
$$

Он локально имеет полный ранг, если строки не параллельны. Это необходимое,
но не достаточное условие практического восстановления параметров.

Расчёт ниже относится к вещественному знаковому $Z$. Для реально используемого
модуля требуется явно зафиксировать оператор $|Z^*|$; совпадение локальных
производных со знаковой ветвью допустимо лишь после проверки фазы, знака и
отсутствия перехода через ноль.

### 2. Точные ограничения постановки

- Если $h$ также неизвестна, пара даёт матрицу $2\times3$ и не может локально
  определить три параметра без дополнительной информации.
- Если неизвестен gain $g$ в $y=gZ$, то для логарифмического наблюдения столбец
  $\partial\ln y/\partial\ln g=1$. Из однородности модели
  $S_{\rho_1}+S_{\rho_2}=1$, поэтому этот столбец равен сумме двух столбцов
  сопротивлений. Абсолютный масштаб $\rho_1,\rho_2$ неидентифицируем при любом
  числе размеров без независимой калибровки или внешнего референса.

### 3. Практическая идентифицируемость

Она требует реальной модели наблюдения, ковариации и профилей функции потерь.
Для последовательных записей ковариация должна учитывать не только остаток
внутри плато, но и неповторяемость состояния, дыхательной задержки, ручной
наклейки, контакта, дрейф, фиксированный порядок, gain/offset и расхождение
плоской модели с анатомией. Эти компоненты пока не оценены и не заменяются
одной условной $\sigma$.


In [ ]:
from pathlib import Path
import platform
import sys

import numpy as np

candidates = [Path.cwd(), Path.cwd() / "Colab Notebooks", Path.cwd().parent]
model_paths = sorted({
    (candidate / "two_layer_model.py").resolve()
    for candidate in candidates
    if (candidate / "two_layer_model.py").is_file()
})
if len(model_paths) != 1:
    raise RuntimeError(f"expected one canonical two_layer_model.py, found: {model_paths}")

model_path = model_paths[0]
sys.path.insert(0, str(model_path.parent))
import two_layer_model as tlm

if Path(tlm.__file__).resolve() != model_path:
    raise RuntimeError(f"imported non-canonical model: {tlm.__file__}")

print(f"Python {platform.python_version()}; NumPy {np.__version__}")
print(f"Модель: {model_path}")


In [ ]:
# Синтетическая рабочая точка из 31.01; не параметры добровольца.
RHO1 = 5.0       # Ом·м
RHO2 = 20.0      # Ом·м
H = 0.020        # м
BETA = 0.5
SIZES_M = np.arange(0.050, 0.141, 0.010)
MODEL_VALIDITY = "unverified_without_subject_specific_CT_FEM_Lmax"

print("status=illustrative_unvalidated")
print("observation_operator=signed_real_Z; experimental_magnitude_not_yet_verified")
print(f"model_validity={MODEL_VALIDITY}")
print(f"rho1={RHO1:g} Ом·м; rho2={RHO2:g} Ом·м; h={H*1000:g} мм; beta={BETA:g}")
print("размеры, мм:", [int(round(value * 1000)) for value in SIZES_M])


In [ ]:
def log_jacobian(sizes_m, rho1=RHO1, rho2=RHO2, h=H, beta=BETA, include_h=False):
    rows = []
    for size_m in np.asarray(sizes_m, dtype=float):
        a, b = tlm.geometry_from_size(float(size_m), beta)
        result = tlm.evaluate(rho1, rho2, h, a, b)
        if result.z == 0:
            raise RuntimeError("logarithmic sensitivities are undefined at Z=0")
        row = [rho1 * result.d_rho1 / result.z, rho2 * result.d_rho2 / result.z]
        if include_h:
            row.append(h * result.d_h / result.z)
        rows.append(row)
    matrix = np.asarray(rows, dtype=float)
    if not np.isfinite(matrix).all():
        raise RuntimeError("non-finite Jacobian")
    return matrix


def pair_metrics(size_i, size_j):
    matrix = log_jacobian([size_i, size_j])
    singular = np.linalg.svd(matrix, compute_uv=False)
    determinant = float(np.linalg.det(matrix))
    row_norms = np.linalg.norm(matrix, axis=1)
    cosine = float(np.dot(matrix[0], matrix[1]) / np.prod(row_norms))
    angle_deg = float(np.degrees(np.arccos(np.clip(cosine, -1.0, 1.0))))
    return {
        "det": determinant,
        "sigma_max": float(singular[0]),
        "sigma_min": float(singular[-1]),
        "condition": float(singular[0] / singular[-1]),
        "row_angle_deg": angle_deg,
    }


def conditional_log_crlb(log_jacobian_matrix, relative_error_covariance):
    jacobian = np.asarray(log_jacobian_matrix, dtype=float)
    covariance = np.asarray(relative_error_covariance, dtype=float)
    if covariance.shape != (jacobian.shape[0], jacobian.shape[0]):
        raise ValueError("covariance shape must match the number of observations")
    if not np.allclose(covariance, covariance.T, rtol=0.0, atol=1e-12):
        raise ValueError("covariance must be symmetric")
    cholesky = np.linalg.cholesky(covariance)
    whitened = np.linalg.solve(cholesky, jacobian)
    singular = np.linalg.svd(whitened, compute_uv=False)
    if singular[-1] <= np.finfo(float).eps * singular[0]:
        raise np.linalg.LinAlgError("whitened Jacobian is rank deficient")
    information = whitened.T @ whitened
    return np.linalg.inv(information)


# Самотесты тождеств и интерфейса; единичная ковариация здесь не модель данных.
full = log_jacobian(SIZES_M)
np.testing.assert_allclose(full[:, 0] + full[:, 1], 1.0, rtol=1e-10, atol=1e-12)
assert np.linalg.matrix_rank(log_jacobian([0.050, 0.050])) == 1
test_covariance = conditional_log_crlb(log_jacobian([0.050, 0.140]), np.eye(2))
assert np.isfinite(test_covariance).all() and np.all(np.diag(test_covariance) > 0)
print("Самотесты якобиана и условного CRLB: пройдены")


In [ ]:
pair_table = []
for i, size_i in enumerate(SIZES_M):
    for size_j in SIZES_M[i + 1:]:
        metrics = pair_metrics(float(size_i), float(size_j))
        pair_table.append({"Li": float(size_i), "Lj": float(size_j), **metrics})

assert len(pair_table) == 45
assert all(row["sigma_min"] > 0 for row in pair_table)

ordered = sorted(pair_table, key=lambda row: row["sigma_min"])
print(f"Число разных пар: {len(ordered)}; все имеют локальный ранг 2 в этой рабочей точке.")
print("Пять наименьших и пять наибольших sigma_min невзвешенного log-J:")
for row in ordered[:5] + ordered[-5:]:
    print(
        f"  {row['Li']*1000:3.0f}-{row['Lj']*1000:3.0f} мм: "
        f"sigma_min={row['sigma_min']:.6f}; cond={row['condition']:.2f}; "
        f"angle={row['row_angle_deg']:.3f}°"
    )

best_idealized = ordered[-1]
print(
    "Максимум sigma_min в этом единственном невзвешенном синтетическом сценарии: "
    f"{best_idealized['Li']*1000:.0f}-{best_idealized['Lj']*1000:.0f} мм."
)
print("Это диагностический результат идеальной модели, не выбор оптимальной пары.")


In [ ]:
reference_pair = [0.050, 0.140]

# Два размера и три неизвестных (ln rho1, ln rho2, ln h): существует null-направление.
pair_with_h = log_jacobian(reference_pair, include_h=True)
_, singular_h, vh_h = np.linalg.svd(pair_with_h, full_matrices=True)
null_h = vh_h[-1]
null_h = null_h / np.linalg.norm(null_h)
np.testing.assert_allclose(pair_with_h @ null_h, 0.0, rtol=0.0, atol=1e-12)
print("Пара 50-140 мм при неизвестном h:")
print("  shape=", pair_with_h.shape, "; rank=", np.linalg.matrix_rank(pair_with_h))
print("  локальное null-направление (ln rho1, ln rho2, ln h)=", np.round(null_h, 6))

# Даже все размеры не снимают точную симметрию абсолютного масштаба при неизвестном gain.
all_two_parameters = log_jacobian(SIZES_M)
with_gain = np.column_stack([all_two_parameters, np.ones(len(SIZES_M))])
scale_null = np.array([1.0, 1.0, -1.0])
np.testing.assert_allclose(with_gain @ scale_null, 0.0, rtol=0.0, atol=1e-10)
print("Все размеры при неизвестном gain:")
print("  shape=", with_gain.shape, "; rank=", np.linalg.matrix_rank(with_gain))
print("  точное null-направление (ln rho1, ln rho2, ln g)=(1, 1, -1)")

# При калиброванном тракте три и более размера могут локально дать ранг 3,
# но это не доказывает практическую определимость или анатомическую верность h.
all_with_h = log_jacobian(SIZES_M, include_h=True)
singular_all_h = np.linalg.svd(all_with_h, compute_uv=False)
print("Все размеры, калиброванный тракт, неизвестные rho1/rho2/h:")
print("  rank=", np.linalg.matrix_rank(all_with_h), "; singular values=", np.round(singular_all_h, 6))


## Что этот расчёт устанавливает — и чего не устанавливает

**Устанавливает только для идеальной модели и выбранной синтетической точки:**

- разные размеры дают непараллельные строки чувствительности, поэтому при
  известном $h$ и калиброванном тракте пара может иметь локальный ранг 2;
- одна пара не определяет одновременно $\rho_1,\rho_2,h$;
- неизвестный gain создаёт точную масштабную симметрию абсолютных
  сопротивлений, которую увеличение числа размеров не устраняет;
- сингулярные числа невзвешенного логарифмического якобиана можно использовать
  как структурную диагностику, но не как самостоятельный критерий выбора.

**Не устанавливает:**

- реальную ковариацию, точность, доверительный интервал или CRLB эксперимента;
- пригодность плоской модели для любого размера без субъектного $L_{max}$;
- оптимальную пару для добровольца или будущего прибора;
- глобальную единственность нелинейной обратной задачи;
- возможность переноса между реокардиомониторами МГТУ и РНЦХ.

Максимум $\sigma_{min}$ в таблице зависит от рабочей точки, параметризации и
весов. Он намеренно не переносится в серию `32` как готовое решение.


## Условия практического продолжения

1. Для экспериментов 2 и 3 отдельно установить оператор наблюдения, gain,
   offset, знак/модуль, частоту и канал.
2. Зафиксировать субъектные принятые размеры после QC: 9 уникальных записей у
   добровольца с поздней копией 100 мм и до 10 независимых записей у второго.
3. Оценить раздельно внутриплатовую вариабельность, межзаписную
   неповторяемость, дыхательную неповторяемость, контакт/переклейку, дрейф и
   систематику порядка. Не объявлять их одной независимой гауссовской ошибкой.
4. В полной разработочной постановке фиксировать $h$ по КТ. В сокращённой
   постановке проверять расчётную $h$ относительно КТ как отдельную обратную
   задачу.
5. Только после этого передать в `conditional_log_crlb` обоснованную ковариацию,
   построить профили функции потерь по всем размерам и выполнить
   leave-one-size-out. Два измерения на два параметра не дают остаточных
   степеней свободы для проверки адекватности модели.

До выполнения этих условий статус практической идентифицируемости —
`blocked_missing_calibration_and_error_model`, а не «параметр определён».
